# Stage 1 — `sample_filtered_data_10000/` in `the-stack-v3-python-fim-data`
**Kaggle notebook — thin controller only. All logic lives in `scripts/build_sample.py`,
`scripts/migrate_sample_to_new_repo.py`, and `src/curation/`.**

The 10,000-file curated Python sample (data-stage-1.md) originally lived in
the flat repo `Rudra-G-23/qwen-coder-python-fim-data`. It now lives in the
structured repo `Rudra-G-23/the-stack-v3-python-fim-data`, under
`sample_filtered_data_10000/{data,checkpoints}/` — see `data-stage-2.md` §1-2
for why (repo name finalized, layout documented).

This notebook:
1. Clones the repo
2. Installs the (CPU-only) curation dependencies
3. Sets HF_TOKEN (+ optional WANDB_API_KEY) from Kaggle Secrets
4. Runs the one-time migration from the old flat repo (idempotent — safe to
   re-run; skips anything already migrated)
5. Repairs any checkpoint JSONs left over from the earlier buggy migration
   run (idempotent — safe to re-run)
6. Runs `scripts/build_sample.py` against the new repo/prefix — a no-op
   today since 10,000/10,000 is already collected, but this is exactly the
   cell a future `target.files` increase would resume from
7. Displays the generated filter report + `metadata.json`

No GPU needed for this stage — see `data-stage-1.md` §4.

In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────────
import os
import shutil
import subprocess
import sys

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch : {BRANCH or 'main'}")
print(f"Requested commit : {COMMIT or '(latest on branch)'}")

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

print(f"\n✓ Repository ready at {REPO_DIR}")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")


In [ ]:
# ── Cell 2: Install dependencies (CPU-only, no torch/unsloth needed here) ───
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "datasets", "pyarrow", "huggingface_hub", "pyyaml", "wandb", "weave",
    ],
    check=True,
)


In [ ]:
# ── Cell 3: Authenticate to Hugging Face (+ optional W&B) ─────────────────────
# HF_TOKEN is stored as a Kaggle Secret — NEVER hardcode tokens.
# Add it: Kaggle account → Settings → Secrets → Add New Secret
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

# W&B tracking is optional here — the scripts fall back to printing a
# warning and skipping W&B logging if this secret isn't set (see
# src/curation/wandb_logger.py / scripts/generate_fim_variants.py's
# init_wandb_run). Add the WANDB_API_KEY secret to enable it.
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("✓ HF_TOKEN and WANDB_API_KEY loaded from Kaggle Secrets")
except Exception:
    print("✓ HF_TOKEN loaded. WANDB_API_KEY not set — W&B tracking will be skipped.")


In [ ]:
# ── Cell 4: One-time migration from the old flat repo (idempotent) ───────────
# Copies the already-collected 10k-file sample's chunk parquet + checkpoint
# JSONs from Rudra-G-23/qwen-coder-python-fim-data into
# sample_filtered_data_10000/{data,checkpoints}/ in the new repo. Safe to
# re-run — skips files that already exist at their destination.
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/migrate_sample_to_new_repo.py"],
    check=True,
)


In [ ]:
# ── Cell 5: Repair checkpoints from the earlier buggy migration ─────────────
# migrate_sample_to_new_repo.py originally uploaded checkpoint JSONs whose
# chunk_file field pointed at the old flat-repo filename instead of the new
# nested sample_filtered_data_10000/data/ path, which makes Cell 6 (build_sample.py)
# 404 on resume. That script is now fixed for future migrations, but
# checkpoints it already uploaded stay broken until patched here.
# Idempotent — safe to re-run; skips any checkpoint already correct.
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/fix_migrated_checkpoint_paths.py"],
    check=True,
)


In [ ]:
# ── Cell 6: Confirm the sample against the new repo/prefix ───────────────────
# configs/data/stack_v3_filter.yaml now points checkpoint.hf_dataset_repo at
# the-stack-v3-python-fim-data with checkpoint.path_prefix =
# sample_filtered_data_10000. Target is already met (10,000/10,000), so this
# resumes, sees that immediately, and just (re)writes the filter report +
# metadata.json from checkpoint history — no new streaming happens.
# MAX_GB is the session's streaming budget (see build_sample.py's docstring):
# a resume/target-already-met run like this one doesn't touch it, but it
# matters as soon as target.files in configs/data/stack_v3_filter.yaml is
# raised and this cell has to stream new files again. Matches the MAX_GB
# convention in notebooks/kaggle/curate_and_checkpoint.ipynb.
import subprocess
import sys

MAX_GB = 1

subprocess.run(
    [sys.executable, "scripts/build_sample.py", "--max-gb", str(MAX_GB)],
    check=True,
)


In [ ]:
# ── Cell 7: Show the filter report + metadata.json ────────────────────────
from IPython.display import Markdown, display
import json

with open(f"{REPO_DIR}/reports/stage1_filter_report.md", encoding="utf-8") as f:
    display(Markdown(f.read()))

with open(f"{REPO_DIR}/reports/sample_filtered_data_10000_metadata.json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))
